# Full GAN → real + synthetic CNN

Set `IMAGE_DIR` to your original ISIC image folder. The workflow uses the verified AK-inclusive GitHub manifest (1,056 train / 353 validation / 352 test), loads the three corresponding CNN initializations, and builds the GAN input locally. There is no pilot, and only the train and validation image contents are read. Run `bash RUN_WORKFLOW.sh preflight` with `ISIC_IMAGE_DIR` set to check the inputs without GPU training. See `START_HERE.md` for the shared GAN/pool and matched Adam 0.0001 baseline.

In [ ]:
from pathlib import Path
import subprocess
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "GAN.sh").is_file():
    candidates = [ROOT / "workflows/full_gan", ROOT.parent / "workflows/full_gan"]
    found = [p for p in candidates if (p / "GAN.sh").is_file()]
    if len(found) != 1:
        raise FileNotFoundError("Open from the downloaded source package or repository")
    ROOT = found[0]
# Set this once to your original ISIC image folder.
IMAGE_DIR = ROOT / "data" / "real"
IMAGE_DIR = IMAGE_DIR.expanduser().resolve()
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError("Set IMAGE_DIR to the existing original dataset folder")
env = os.environ.copy()
env["ISIC_IMAGE_DIR"] = str(IMAGE_DIR)
# If this notebook was opened from a different working directory, set ROOT to
# the extracted ISIC_Full_GAN_Workflows directory.
if not (ROOT / "GAN.sh").is_file():
    raise FileNotFoundError("Open this notebook from the complete extracted workflow folder")
process = subprocess.Popen(["bash", str(ROOT / "GAN.sh")], cwd=ROOT, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
except BaseException:
    process.terminate()
    process.wait()
    raise
if returncode:
    raise RuntimeError("Workflow stopped. Read the message above; partial outputs are preserved.")
